In [ ]:
import numpy as np
import pandas as pd
import random
import joblib
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Bidirectional, GlobalAveragePooling1D, BatchNormalization, Conv1D, MaxPooling1D, Attention, Masking, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
import os
import tensorflow as tf

# --- TIME-AWARE FEATURE EXTRACTOR ---
def build_features(df):
    df = df.copy()
    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)

    # Calculate Time Delta
    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)

    df['accel_x'] = df.groupby('track_no')['vel_x'].diff() / df['dt']
    df['accel_y'] = df.groupby('track_no')['vel_y'].diff() / df['dt']
    df['accel_z'] = df.groupby('track_no')['vel_z'].diff() / df['dt']
    df[['accel_x', 'accel_y', 'accel_z']] = df[['accel_x', 'accel_y', 'accel_z']].fillna(0)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / df['dt']
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / df['dt']
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / df['dt']
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['curvature'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2) / (df['range_mag'] + 1e-8)
    df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360

    def calc_turn(group):
        v = group[['vel_x', 'vel_y', 'vel_z']].values
        dt = group['dt'].values
        res = [0.0]
        for i in range(1, len(v)):
            n1, n2 = np.linalg.norm(v[i-1]), np.linalg.norm(v[i])
            if n1>0 and n2>0 and dt[i]>0:
                cos_ang = np.clip(np.dot(v[i-1], v[i])/(n1*n2), -1, 1)
                res.append(np.degrees(np.arccos(cos_ang))/dt[i])
            else: res.append(0.0)
        return np.array(res, dtype=np.float64)

    df['velocity_turn_rate'] = df.groupby('track_no').apply(calc_turn).reset_index(level=0, drop=True)

    cols_to_convert = ['accel_mag', 'velocity_turn_rate', 'z_ratio', 'dt']
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_turn'] = df.groupby('track_no')['velocity_turn_rate'].cummax()
    df['cum_max_z_ratio'] = df.groupby('track_no')['z_ratio'].cummax()
    df['cum_min_z_ratio'] = df.groupby('track_no')['z_ratio'].cummin()

    df['accel_mag_rolling_mean'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
    df['accel_mag_rolling_std'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).std().reset_index(level=0, drop=True)

    def calc_dir_change(group):
        h = group['heading'].values
        res = [0.0]
        for i in range(1, len(h)):
            delta = abs((h[i] - h[i-1]) % 360)
            res.append(1.0 if min(delta, 360-delta) > 45 else 0.0)
        return np.array(res, dtype=np.float64)

    df['direction_change_rate'] = df.groupby('track_no').apply(calc_dir_change).reset_index(level=0, drop=True)
    df['direction_change_rate'] = pd.to_numeric(df['direction_change_rate'], errors='coerce').fillna(0.0)

    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0.0)

    return df.fillna(0.0)

# --- ADVANCED TRAINING SIMULATOR (Identical to your stress test) ---
def generate_complex_trajectory(track_class, duration=80, dt=10.0):
    pos_x_list, pos_y_list, pos_z_list = [], [], []
    vel_x_list, vel_y_list, vel_z_list = [], [], []
    track_no_list, track_class_list, ntp_sec_list = [], [], []

    track_id = random.randint(100000, 999999)

    spawn_radius = random.uniform(10000, 80000)
    spawn_angle = random.uniform(0, 2 * np.pi)
    pos_x = spawn_radius * np.cos(spawn_angle)
    pos_y = spawn_radius * np.sin(spawn_angle)

    if track_class == "CIVILIAN":
        pos_z = np.random.uniform(5000, 12000)
        v_mag = np.random.uniform(220, 310)
    elif track_class == "MISSILE":
        pos_z = np.random.uniform(50, 500) if random.random() < 0.3 else np.random.uniform(8000, 22000)
        v_mag = np.random.uniform(400, 1100)
    elif track_class == "HELICOPTER":
        pos_z = np.random.uniform(20, 1500)
        v_mag = np.random.uniform(0, 85)
    elif track_class == "FIGHTER_JET":
        pos_z = np.random.uniform(500, 19000)
        v_mag = np.random.uniform(250, 750)
    elif track_class == "SKIFF":
        pos_z = 0.0
        v_mag = np.random.uniform(2, 28)

    angle = random.uniform(0, 2 * np.pi)
    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])

    start_time = 3991182426
    actual_elapsed_time = 0.0

    maneuver_timer = 0
    maneuver_type = "NONE"
    target_turn_rate = 0.0
    in_burst_loss = False

    for t in range(duration):
        if maneuver_timer <= 0:
            if track_class == "FIGHTER_JET" and random.random() < 0.15:
                maneuver_timer = random.randint(4, 10)
                maneuver_type = random.choice(["EVASIVE_TURN", "SPLIT_S_DIVE"])
                target_turn_rate = np.radians(random.uniform(-15, 15))
            elif track_class == "CIVILIAN" and random.random() < 0.10:
                maneuver_timer = random.randint(15, 30)
                maneuver_type = "HOLDING_PATTERN"
                target_turn_rate = np.radians(random.choice([-3, 3]))
            elif track_class == "HELICOPTER" and random.random() < 0.20:
                maneuver_timer = random.randint(3, 8)
                maneuver_type = random.choice(["POP_UP", "SUDDEN_HOVER"])
            elif track_class == "SKIFF" and random.random() < 0.25:
                maneuver_timer = random.randint(2, 6)
                maneuver_type = "ZIG_ZAG"
                target_turn_rate = np.radians(random.uniform(-25, 25))
            elif track_class == "MISSILE" and random.random() < 0.10:
                maneuver_timer = random.randint(3, 5)
                maneuver_type = "TERMINAL_POP_UP"
            else:
                maneuver_type = "CRUISE"

        if maneuver_timer > 0:
            maneuver_timer -= 1
            if maneuver_type in ["EVASIVE_TURN", "HOLDING_PATTERN", "ZIG_ZAG"]:
                c, s = np.cos(target_turn_rate * dt), np.sin(target_turn_rate * dt)
                vel[0], vel[1] = vel[0]*c - vel[1]*s, vel[0]*s + vel[1]*c
            elif maneuver_type == "SPLIT_S_DIVE":
                vel[2] -= 40 * dt
                vel[0] *= 1.05; vel[1] *= 1.05
            elif maneuver_type == "POP_UP":
                vel[2] = 25
                vel[0] *= 0.5; vel[1] *= 0.5
            elif maneuver_type == "SUDDEN_HOVER":
                vel *= 0.5
            elif maneuver_type == "TERMINAL_POP_UP":
                vel[2] = 50
        else:
            if track_class != "CIVILIAN":
                vel += np.random.normal(0, 1.5, 3)
            else:
                vel += np.random.normal(0, 0.2, 3)
                vel[2] *= 0.95

        if track_class == "SKIFF": vel[2] = 0.0

        pos_x += vel[0] * dt
        pos_y += vel[1] * dt
        pos_z += vel[2] * dt

        if pos_z < 0 and track_class != "SKIFF":
            pos_z = 0.0
            vel[2] = 0.0
        if track_class == "SKIFF": pos_z = 0.0

        actual_elapsed_time += dt

        if in_burst_loss:
            if random.random() < 0.60: in_burst_loss = False
            else: continue
        else:
            if random.random() < 0.08:
                in_burst_loss = True
                continue

        r_mag = np.sqrt(pos_x**2 + pos_y**2 + pos_z**2)
        noise_multiplier = max(1.0, (r_mag / 25000) ** 1.5)

        pos_x_noisy = pos_x + np.random.normal(0, 15 * noise_multiplier)
        pos_y_noisy = pos_y + np.random.normal(0, 15 * noise_multiplier)
        if track_class == "SKIFF": pos_z_noisy = abs(np.random.normal(0, 8 * noise_multiplier))
        else: pos_z_noisy = pos_z + np.random.normal(0, 20 * noise_multiplier)

        vel_x_noisy = vel[0] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_y_noisy = vel[1] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_z_noisy = vel[2] + np.random.normal(0, 1.5 * noise_multiplier)

        timestamp = start_time + int(actual_elapsed_time)

        pos_x_list.append(pos_x_noisy); pos_y_list.append(pos_y_noisy); pos_z_list.append(pos_z_noisy)
        vel_x_list.append(vel_x_noisy); vel_y_list.append(vel_y_noisy); vel_z_list.append(vel_z_noisy)
        track_no_list.append(track_id); track_class_list.append(track_class); ntp_sec_list.append(timestamp)

    df_data = {
        'pos_x': pos_x_list, 'pos_y': pos_y_list, 'pos_z': pos_z_list,
        'vel_x': vel_x_list, 'vel_y': vel_y_list, 'vel_z': vel_z_list,
        'track_no': track_no_list, 'track_class': track_class_list, 'ntp_sec': ntp_sec_list
    }
    return pd.DataFrame(df_data)

def generate_dataset(num_samples=250, duration=80):
    dfs = []
    track_classes = ["FIGHTER_JET", "SKIFF", "MISSILE", "HELICOPTER", "CIVILIAN"]
    for class_choice in track_classes:
        for _ in range(num_samples):
            df = generate_complex_trajectory(class_choice, duration=duration)
            if df is not None and len(df) > 5:
                dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# --- ARCHITECTURE ---
def build_robust_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    masked = Masking(mask_value=0.0)(inputs)

    conv1 = Conv1D(32, kernel_size=3, activation='relu', padding='same')(masked)
    conv2 = Conv1D(32, kernel_size=5, activation='relu', padding='same')(masked)
    conv3 = Conv1D(32, kernel_size=7, activation='relu', padding='same')(masked)

    merged = concatenate([conv1, conv2, conv3])
    merged = BatchNormalization()(merged)
    merged = MaxPooling1D(pool_size=2)(merged)

    lstm_out = Bidirectional(LSTM(64, return_sequences=True, dropout=0.2))(merged)

    attention = Attention()([lstm_out, lstm_out])
    attention_pooled = GlobalAveragePooling1D()(attention)

    second_lstm = Bidirectional(LSTM(32, return_sequences=False))(lstm_out)

    combined = concatenate([attention_pooled, second_lstm])
    dense = Dense(64, activation='relu')(combined)
    dense = BatchNormalization()(dense)
    dense = Dropout(0.4)(dense)

    outputs = Dense(num_classes, activation='softmax')(dense)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

if __name__ == "__main__":
    if not os.path.exists("trained_model"): os.makedirs("trained_model")
    print("Generating complex real-world training data...")
    df = generate_dataset(num_samples=300) # Generating 1500 total tracks for better generalization
    print("Building memory features...")
    df_enhanced = build_features(df)

    # CRITICAL ADDITION: 'dt' is now explicitly passed to the network
    feature_cols = [
        'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'vel_mag', 'range_mag', 'curvature',
        'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
        'cum_max_z_ratio', 'cum_min_z_ratio',
        'vertical_accel_ratio', 'speed_change_rate'
    ]

    scaler = StandardScaler()
    df_enhanced[feature_cols] = scaler.fit_transform(df_enhanced[feature_cols])
    joblib.dump(scaler, "trained_model/feature_scaler.pkl")

    seq_length = 50
    track_sequences, track_labels = [], []
    print("Building progressive sequences...")

    for track_id, group in df_enhanced.groupby('track_no'):
        data_matrix = group.sort_values('ntp_sec')[feature_cols].values

        for i in range(1, len(data_matrix) + 1):
            if i <= seq_length:
                window = data_matrix[:i]
                padded = np.pad(window, ((seq_length - len(window), 0), (0, 0)), 'constant')
                track_sequences.append(padded)
            else:
                window = data_matrix[i-seq_length:i]
                track_sequences.append(window)
            track_labels.append(group['track_class'].iloc[0])

    X, y = np.array(track_sequences), np.array(track_labels)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    joblib.dump(le, "trained_model/label_encoder.pkl")

    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
    model = build_robust_model((X.shape[1], X.shape[2]), len(le.classes_))

    callbacks = [EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
                 ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)]

    print("Training Next-Level Model...")
    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=64, callbacks=callbacks)
    model.save("trained_model/radar_target_classifier.keras")
    print("Training Complete!")

Generating complex real-world training data...
Building memory features...
Building progressive sequences...
Training Next-Level Model...
Epoch 1/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 55s 38ms/step - accuracy: 0.9406 - loss: 0.1521 - val_accuracy: 0.9701 - val_loss: 0.0702 - learning_rate: 0.0010
Epoch 2/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 51s 38ms/step - accuracy: 0.9703 - loss: 0.0729 - val_accuracy: 0.9774 - val_loss: 0.0504 - learning_rate: 0.0010
Epoch 3/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 47s 36ms/step - accuracy: 0.9758 - loss: 0.0578 - val_accuracy: 0.9839 - val_loss: 0.0375 - learning_rate: 0.0010
Epoch 4/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 48s 36ms/step - accuracy: 0.9793 - loss: 0.0510 - val_accuracy: 0.9846 - val_loss: 0.0347 - learning_rate: 0.0010
Epoch 5/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 48s 36ms/step - accuracy: 0.9811 - loss: 0.0462 - val_accuracy: 0.9862 - val_loss: 0.0323 - learning_rate: 0.0010
Epoch 6/25
1328/1328 ━━━━━━━━━━━━━━━━━━━━ 48s 36ms/step - accuracy: 0.9827 - loss:

2

In [ ]:
import numpy as np
import pandas as pd
import random

def generate_complex_trajectory(track_class, duration=60, dt=10.0):
    pos_x_list, pos_y_list, pos_z_list = [], [], []
    vel_x_list, vel_y_list, vel_z_list = [], [], []
    track_no_list, track_class_list, ntp_sec_list = [], [], []

    track_id = random.randint(100000, 999999)

    # Spawn targets at varying ranges to test range-dependent noise
    spawn_radius = random.uniform(10000, 80000)
    spawn_angle = random.uniform(0, 2 * np.pi)
    pos_x = spawn_radius * np.cos(spawn_angle)
    pos_y = spawn_radius * np.sin(spawn_angle)

    # Overlapping boundary initializations
    if track_class == "CIVILIAN":
        pos_z = np.random.uniform(5000, 12000)
        v_mag = np.random.uniform(220, 310)
    elif track_class == "MISSILE":
        # 30% chance to be a "Sea Skimmer" missile (very low altitude)
        pos_z = np.random.uniform(50, 500) if random.random() < 0.3 else np.random.uniform(8000, 22000)
        v_mag = np.random.uniform(400, 1100)
    elif track_class == "HELICOPTER":
        pos_z = np.random.uniform(20, 1500)
        v_mag = np.random.uniform(0, 85) # Can start in a hover (0 m/s)
    elif track_class == "FIGHTER_JET":
        pos_z = np.random.uniform(500, 19000)
        v_mag = np.random.uniform(250, 750)
    elif track_class == "SKIFF":
        pos_z = 0.0
        v_mag = np.random.uniform(2, 28)

    angle = random.uniform(0, 2 * np.pi)
    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])

    start_time = 3991182426
    actual_elapsed_time = 0.0

    # State Machine Variables for Sustained Maneuvers
    maneuver_timer = 0
    maneuver_type = "NONE"
    target_turn_rate = 0.0

    # Sensor failure simulation variables
    in_burst_loss = False

    for t in range(duration):
        # ---------------------------------------------------
        # 1. SUSTAINED MANEUVER LOGIC (The "Real World" aspect)
        # ---------------------------------------------------
        if maneuver_timer <= 0:
            # Chance to start a new maneuver
            if track_class == "FIGHTER_JET" and random.random() < 0.15:
                maneuver_timer = random.randint(4, 10) # Lasts 40 to 100 seconds
                maneuver_type = random.choice(["EVASIVE_TURN", "SPLIT_S_DIVE"])
                target_turn_rate = np.radians(random.uniform(-15, 15)) # Deg per second

            elif track_class == "CIVILIAN" and random.random() < 0.10:
                maneuver_timer = random.randint(15, 30) # Long holding pattern
                maneuver_type = "HOLDING_PATTERN"
                target_turn_rate = np.radians(random.choice([-3, 3])) # Standard rate turn

            elif track_class == "HELICOPTER" and random.random() < 0.20:
                maneuver_timer = random.randint(3, 8)
                maneuver_type = random.choice(["POP_UP", "SUDDEN_HOVER"])

            elif track_class == "SKIFF" and random.random() < 0.25:
                maneuver_timer = random.randint(2, 6)
                maneuver_type = "ZIG_ZAG"
                target_turn_rate = np.radians(random.uniform(-25, 25))

            elif track_class == "MISSILE" and random.random() < 0.10:
                maneuver_timer = random.randint(3, 5)
                maneuver_type = "TERMINAL_POP_UP" # Pops up before diving
            else:
                maneuver_type = "CRUISE"

        # Execute active maneuver
        if maneuver_timer > 0:
            maneuver_timer -= 1

            if maneuver_type in ["EVASIVE_TURN", "HOLDING_PATTERN", "ZIG_ZAG"]:
                # Apply sustained turning over dt
                c, s = np.cos(target_turn_rate * dt), np.sin(target_turn_rate * dt)
                vel[0], vel[1] = vel[0]*c - vel[1]*s, vel[0]*s + vel[1]*c

            elif maneuver_type == "SPLIT_S_DIVE":
                vel[2] -= 40 * dt # Massive sustained dive
                vel[0] *= 1.05; vel[1] *= 1.05 # Accelerate in dive

            elif maneuver_type == "POP_UP":
                vel[2] = 25 # Rapid climb
                vel[0] *= 0.5; vel[1] *= 0.5 # Lose horizontal speed

            elif maneuver_type == "SUDDEN_HOVER":
                vel *= 0.5 # Bleed speed rapidly to 0

            elif maneuver_type == "TERMINAL_POP_UP":
                vel[2] = 50 # Sharp climb

        else:
            # Normal Cruising - Add slight natural drift
            if track_class != "CIVILIAN":
                vel += np.random.normal(0, 1.5, 3)
            else:
                vel += np.random.normal(0, 0.2, 3)
                vel[2] *= 0.95 # Civil flights tend to strictly self-level

        # Strict Boundaries
        if track_class == "SKIFF":
            vel[2] = 0.0

        # ---------------------------------------------------
        # 2. PHYSICS UPDATE
        # ---------------------------------------------------
        pos_x += vel[0] * dt
        pos_y += vel[1] * dt
        pos_z += vel[2] * dt

        # Hard ground floor
        if pos_z < 0 and track_class != "SKIFF":
            pos_z = 0.0
            vel[2] = 0.0
        if track_class == "SKIFF": pos_z = 0.0

        actual_elapsed_time += dt

        # ---------------------------------------------------
        # 3. BURST PACKET LOSS SIMULATION
        # ---------------------------------------------------
        if in_burst_loss:
            if random.random() < 0.60: # 60% chance to recover radar lock
                in_burst_loss = False
            else:
                continue # Skip appending! Model receives a massive time gap next packet.
        else:
            if random.random() < 0.08: # 8% chance to lose radar lock
                in_burst_loss = True
                continue

        # ---------------------------------------------------
        # 4. RANGE-DEPENDENT SENSOR NOISE
        # ---------------------------------------------------
        r_mag = np.sqrt(pos_x**2 + pos_y**2 + pos_z**2)
        noise_multiplier = max(1.0, (r_mag / 25000) ** 1.5) # Noise scales exponentially past 25km

        pos_x_noisy = pos_x + np.random.normal(0, 15 * noise_multiplier)
        pos_y_noisy = pos_y + np.random.normal(0, 15 * noise_multiplier)

        if track_class == "SKIFF":
            # Sea clutter / wave bounce
            pos_z_noisy = abs(np.random.normal(0, 8 * noise_multiplier))
        else:
            pos_z_noisy = pos_z + np.random.normal(0, 20 * noise_multiplier)

        vel_x_noisy = vel[0] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_y_noisy = vel[1] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_z_noisy = vel[2] + np.random.normal(0, 1.5 * noise_multiplier)

        # ---------------------------------------------------
        # 5. RECORD DATA
        # ---------------------------------------------------
        timestamp = start_time + int(actual_elapsed_time)

        pos_x_list.append(pos_x_noisy)
        pos_y_list.append(pos_y_noisy)
        pos_z_list.append(pos_z_noisy)
        vel_x_list.append(vel_x_noisy)
        vel_y_list.append(vel_y_noisy)
        vel_z_list.append(vel_z_noisy)
        track_no_list.append(track_id)
        track_class_list.append(track_class)
        ntp_sec_list.append(timestamp)

    # Note: Because of burst drop, the lists might be shorter than `duration`. That's intended!
    df_data = {
        'pos_x': pos_x_list, 'pos_y': pos_y_list, 'pos_z': pos_z_list,
        'vel_x': vel_x_list, 'vel_y': vel_y_list, 'vel_z': vel_z_list,
        'track_no': track_no_list, 'track_class': track_class_list, 'ntp_sec': ntp_sec_list
    }
    return pd.DataFrame(df_data)

def generate_stress_test_dataset(num_samples=150, duration=80):
    dfs = []
    track_classes = ["FIGHTER_JET", "SKIFF", "MISSILE", "HELICOPTER", "CIVILIAN"]
    samples_per_class = num_samples // len(track_classes)

    for class_choice in track_classes:
        for _ in range(samples_per_class):
            df = generate_complex_trajectory(class_choice, duration=duration)
            if df is not None and len(df) > 5: # Only keep if at least 5 packets survived the burst loss
                dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

if __name__ == "__main__":
    print("Initializing Real-World Radar Stress Test Generator...")
    # Generating 150 tracks, tracked over 80 potential scans (800 seconds max)
    test_df = generate_stress_test_dataset(num_samples=150, duration=80)
    test_df.to_csv("test_radar_data.csv", index=False)
    print(f"Successfully generated advanced stress-test dataset with {len(test_df)} packets.")
    print("Features injected:")
    print(" - Burst packet loss (Time jumps > 10s)")
    # Print max dt to prove time jumps exist
    test_df['dt'] = test_df.groupby('track_no')['ntp_sec'].diff()
    max_jump = test_df['dt'].max()
    print(f" - Maximum sensor blackout period detected: {max_jump} seconds")
    print(" - Range-dependent exponentially scaling noise")
    print(" - Sustained Holding Patterns, Evasive Dives, and Hovering Maneuvers")

Initializing Real-World Radar Stress Test Generator...
Successfully generated advanced stress-test dataset with 10623 packets.
Features injected:
 - Burst packet loss (Time jumps > 10s)
 - Maximum sensor blackout period detected: 110.0 seconds
 - Range-dependent exponentially scaling noise
 - Sustained Holding Patterns, Evasive Dives, and Hovering Maneuvers


3

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import joblib

def build_features(df):
    df = df.copy()
    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)

    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)

    df['accel_x'] = df.groupby('track_no')['vel_x'].diff() / df['dt']
    df['accel_y'] = df.groupby('track_no')['vel_y'].diff() / df['dt']
    df['accel_z'] = df.groupby('track_no')['vel_z'].diff() / df['dt']
    df[['accel_x', 'accel_y', 'accel_z']] = df[['accel_x', 'accel_y', 'accel_z']].fillna(0)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / df['dt']
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / df['dt']
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / df['dt']
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['curvature'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2) / (df['range_mag'] + 1e-8)
    df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360

    def calc_turn(group):
        v = group[['vel_x', 'vel_y', 'vel_z']].values
        dt = group['dt'].values
        res = [0.0]
        for i in range(1, len(v)):
            n1, n2 = np.linalg.norm(v[i-1]), np.linalg.norm(v[i])
            if n1>0 and n2>0 and dt[i]>0:
                cos_ang = np.clip(np.dot(v[i-1], v[i])/(n1*n2), -1, 1)
                res.append(np.degrees(np.arccos(cos_ang))/dt[i])
            else: res.append(0.0)
        return np.array(res, dtype=np.float64)

    df['velocity_turn_rate'] = df.groupby('track_no').apply(calc_turn).reset_index(level=0, drop=True)

    cols_to_convert = ['accel_mag', 'velocity_turn_rate', 'z_ratio', 'dt']
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_turn'] = df.groupby('track_no')['velocity_turn_rate'].cummax()
    df['cum_max_z_ratio'] = df.groupby('track_no')['z_ratio'].cummax()
    df['cum_min_z_ratio'] = df.groupby('track_no')['z_ratio'].cummin()

    df['accel_mag_rolling_mean'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
    df['accel_mag_rolling_std'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).std().reset_index(level=0, drop=True)

    def calc_dir_change(group):
        h = group['heading'].values
        res = [0.0]
        for i in range(1, len(h)):
            delta = abs((h[i] - h[i-1]) % 360)
            res.append(1.0 if min(delta, 360-delta) > 45 else 0.0)
        return np.array(res, dtype=np.float64)

    df['direction_change_rate'] = df.groupby('track_no').apply(calc_dir_change).reset_index(level=0, drop=True)
    df['direction_change_rate'] = pd.to_numeric(df['direction_change_rate'], errors='coerce').fillna(0.0)

    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0.0)

    return df.fillna(0.0)

def predict_single_track(track_group, model, le, scaler):
    feature_cols = [
        'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'vel_mag', 'range_mag', 'curvature',
        'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
        'cum_max_z_ratio', 'cum_min_z_ratio',
        'vertical_accel_ratio', 'speed_change_rate'
    ]

    try:
        features = track_group[feature_cols].values
        features_scaled = scaler.transform(features)

        seq_length = 50
        if len(features_scaled) >= seq_length:
            input_data = features_scaled[-seq_length:].reshape(1, seq_length, -1)
        else:
            padded = np.pad(features_scaled, ((seq_length - len(features_scaled), 0), (0, 0)), 'constant')
            input_data = padded.reshape(1, seq_length, -1)

        pred_probs = model.predict(input_data, verbose=0)[0]
        predicted_class_idx = np.argmax(pred_probs)
        confidence = np.max(pred_probs)
        predicted_label = le.inverse_transform([predicted_class_idx])[0]

        return {'prediction': predicted_label, 'confidence': float(confidence)}
    except Exception as e:
        return {'prediction': f'ERROR: {str(e)}', 'confidence': 0.0}

def simulate_streaming_radar(df_test, model, le, scaler):
    print("\n" + "="*60)
    print(" REAL-TIME STREAMING SIMULATION (Packet-by-Packet)")
    print("="*60)

    sample_tracks = df_test['track_no'].unique()[:3]
    for tid in sample_tracks:
        track_data = df_test[df_test['track_no'] == tid].sort_values('ntp_sec')
        actual = track_data['track_class'].iloc[0]
        print(f"\n📡 Tracking ID {tid} (Actual Target: {actual})")

        checkpoints = [1, 3, 10, 25, len(track_data)]
        for i in checkpoints:
            if i > len(track_data): continue

            current_stream = track_data.iloc[:i]
            res = predict_single_track(current_stream, model, le, scaler)

            status = "✅" if res['prediction'] == actual else "⏳"
            print(f"  Packets received: {i:2d} -> Predicted: {res['prediction']:12s} | Confidence: {res['confidence']:.2%} {status}")

def process_test_file(file_path):
    print("Loading trained model...")
    model = load_model("trained_model/radar_target_classifier.keras", compile=False)
    le = joblib.load("trained_model/label_encoder.pkl")
    scaler = joblib.load("trained_model/feature_scaler.pkl")
    print("✓ Model and encoders loaded successfully")

    df_test = pd.read_csv(file_path).dropna(subset=['track_no'])
    print("Building cumulative memory features...")
    df_test = build_features(df_test)

    simulate_streaming_radar(df_test, model, le, scaler)

    print("\nProcessing final full sequences for report...")
    final_results = []
    for track_id, group in df_test.groupby('track_no'):
        actual_label = group['track_class'].iloc[0]
        pred_res = predict_single_track(group, model, le, scaler)
        final_results.append({
            'Track ID': track_id, 'Actual Class': actual_label, 'Predicted Class': pred_res['prediction'],
            'Confidence': f"{pred_res['confidence']:.4f}", 'Result': '✅ Correct' if actual_label == pred_res['prediction'] else '❌ Wrong'
        })
    return final_results

def visualize_results_table(results):
    df_res = pd.DataFrame(results)
    correct_count = len(df_res[df_res['Result'] == '✅ Correct'])
    total = len(df_res)
    accuracy = (correct_count / total) * 100 if total > 0 else 0

    print("\n" + "="*60)
    print(f" FINAL CLASSIFICATION REPORT ")
    print("="*60)
    print(f"Total Tracks: {total} | Correct: {correct_count} | Accuracy: {accuracy:.2f}%")
    print("-" * 60)
    print(df_res.to_string(index=False))
    print("="*60)

if __name__ == "__main__":
    results_list = process_test_file("test_radar_data.csv")
    if results_list: visualize_results_table(results_list)

Loading trained model...
✓ Model and encoders loaded successfully
Building cumulative memory features...

 REAL-TIME STREAMING SIMULATION (Packet-by-Packet)

📡 Tracking ID 109048 (Actual Target: MISSILE)
  Packets received:  1 -> Predicted: MISSILE      | Confidence: 99.83% ✅
  Packets received:  3 -> Predicted: MISSILE      | Confidence: 100.00% ✅
  Packets received: 10 -> Predicted: MISSILE      | Confidence: 100.00% ✅
  Packets received: 25 -> Predicted: MISSILE      | Confidence: 100.00% ✅
  Packets received: 73 -> Predicted: MISSILE      | Confidence: 100.00% ✅

📡 Tracking ID 114785 (Actual Target: HELICOPTER)
  Packets received:  1 -> Predicted: HELICOPTER   | Confidence: 96.29% ✅
  Packets received:  3 -> Predicted: HELICOPTER   | Confidence: 98.72% ✅
  Packets received: 10 -> Predicted: HELICOPTER   | Confidence: 99.70% ✅
  Packets received: 25 -> Predicted: HELICOPTER   | Confidence: 100.00% ✅
  Packets received: 73 -> Predicted: HELICOPTER   | Confidence: 100.00% ✅

📡 Trackin

In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import joblib
import os

# --- SMART PREPROCESSING BRIDGE ---
def robust_feature_extractor(df):
    """
    A smart preprocessing bridge that handles missing columns from various real-world
    radar ICDs. It uses sensor-provided data if available, and mathematically derives
    the rest to ensure the model always gets its exact required 18 features.
    """
    df = df.copy()

    # 1. Ensure basic sorting and identifiers
    if 'track_no' not in df.columns:
        df['track_no'] = 1  # Fallback if no track ID is provided
    if 'ntp_sec' not in df.columns:
        # Fallback fake time if timestamps are completely missing
        df['ntp_sec'] = np.arange(len(df)) * 10.0

    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)

    # 2. Derive Time Delta (dt)
    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)

    # 3. Smart Base Kinematics (Use sensor data if exists, else derive)
    for axis in ['x', 'y', 'z']:
        if f'vel_{axis}' not in df.columns:
            df[f'vel_{axis}'] = df.groupby('track_no')[f'pos_{axis}'].diff() / df['dt']
            df[f'vel_{axis}'] = df[f'vel_{axis}'].fillna(0)

    for axis in ['x', 'y', 'z']:
        if f'accel_{axis}' not in df.columns:
            df[f'accel_{axis}'] = df.groupby('track_no')[f'vel_{axis}'].diff() / df['dt']
            df[f'accel_{axis}'] = df[f'accel_{axis}'].fillna(0)

    if 'accel_mag' not in df.columns:
        df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    # 4. Derive Jerk (Always missing in standard ICDs, must calculate from accel)
    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / df['dt']
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / df['dt']
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / df['dt']
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)

    # 5. Magnitudes and Angles
    if 'vel_mag' not in df.columns:
        df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    if 'range_mag' not in df.columns:
        df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    if 'heading' not in df.columns:
        df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360

    # 6. Model-Specific Ratios
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['curvature'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2) / (df['range_mag'] + 1e-8)
    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0.0)

    # 7. Turn Rates
    def calc_turn(group):
        v = group[['vel_x', 'vel_y', 'vel_z']].values
        dt = group['dt'].values
        res = [0.0]
        for i in range(1, len(v)):
            n1, n2 = np.linalg.norm(v[i-1]), np.linalg.norm(v[i])
            if n1 > 0 and n2 > 0 and dt[i] > 0:
                cos_ang = np.clip(np.dot(v[i-1], v[i])/(n1*n2), -1, 1)
                res.append(np.degrees(np.arccos(cos_ang))/dt[i])
            else:
                res.append(0.0)
        return np.array(res, dtype=np.float64)

    df['velocity_turn_rate'] = df.groupby('track_no').apply(calc_turn).reset_index(level=0, drop=True)

    # Clean numeric types before cumulative math
    cols_to_convert = ['accel_mag', 'velocity_turn_rate', 'z_ratio', 'dt']
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    # 8. Cumulative Memory Features
    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_turn'] = df.groupby('track_no')['velocity_turn_rate'].cummax()
    df['cum_max_z_ratio'] = df.groupby('track_no')['z_ratio'].cummax()
    df['cum_min_z_ratio'] = df.groupby('track_no')['z_ratio'].cummin()

    df = df.fillna(0.0)

    # 9. Strict column enforcement
    required_features = [
        'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'vel_mag', 'range_mag', 'curvature',
        'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
        'cum_max_z_ratio', 'cum_min_z_ratio',
        'vertical_accel_ratio', 'speed_change_rate'
    ]

    for col in required_features:
        if col not in df.columns:
            df[col] = 0.0

    return df

# --- PREDICTION LOGIC ---
def predict_single_track(track_group, model, le, scaler):
    feature_cols = [
        'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'vel_mag', 'range_mag', 'curvature',
        'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
        'cum_max_z_ratio', 'cum_min_z_ratio',
        'vertical_accel_ratio', 'speed_change_rate'
    ]

    try:
        features = track_group[feature_cols].values
        features_scaled = scaler.transform(features)

        seq_length = 50
        if len(features_scaled) >= seq_length:
            input_data = features_scaled[-seq_length:].reshape(1, seq_length, -1)
        else:
            padded = np.pad(features_scaled, ((seq_length - len(features_scaled), 0), (0, 0)), 'constant')
            input_data = padded.reshape(1, seq_length, -1)

        pred_probs = model.predict(input_data, verbose=0)[0]
        predicted_class_idx = np.argmax(pred_probs)
        confidence = np.max(pred_probs)
        predicted_label = le.inverse_transform([predicted_class_idx])[0]

        return {'prediction': predicted_label, 'confidence': float(confidence)}
    except Exception as e:
        return {'prediction': f'ERROR: {str(e)}', 'confidence': 0.0}

def simulate_streaming_radar(df_test, model, le, scaler):
    print("\n" + "="*60)
    print(" REAL-TIME STREAMING SIMULATION (Packet-by-Packet)")
    print("="*60)

    sample_tracks = df_test['track_no'].unique()[:3]
    for tid in sample_tracks:
        track_data = df_test[df_test['track_no'] == tid].sort_values('ntp_sec')
        # Check if actual class is available in the test file (it might not be in pure raw data)
        actual = track_data['track_class'].iloc[0] if 'track_class' in track_data.columns else "UNKNOWN"
        print(f"\n📡 Tracking ID {tid} (Actual Target: {actual})")

        checkpoints = [1, 3, 10, 25, len(track_data)]
        for i in checkpoints:
            if i > len(track_data): continue

            current_stream = track_data.iloc[:i]
            res = predict_single_track(current_stream, model, le, scaler)

            status = "✅" if res['prediction'] == actual else ("⏳" if actual != "UNKNOWN" else "🔍")
            print(f"  Packets received: {i:2d} -> Predicted: {res['prediction']:12s} | Confidence: {res['confidence']:.2%} {status}")

def process_test_file(file_path):
    print(f"Loading trained model and testing against: {file_path}")
    model = load_model("trained_model/radar_target_classifier.keras", compile=False)
    le = joblib.load("trained_model/label_encoder.pkl")
    scaler = joblib.load("trained_model/feature_scaler.pkl")
    print("✓ Model and encoders loaded successfully")

    df_test = pd.read_csv(file_path)

    # Drop rows without a track number if it exists
    if 'track_no' in df_test.columns:
        df_test = df_test.dropna(subset=['track_no'])

    print("Extracting and bridging features...")
    df_test = robust_feature_extractor(df_test)

    simulate_streaming_radar(df_test, model, le, scaler)

    print("\nProcessing final full sequences for report...")
    final_results = []
    for track_id, group in df_test.groupby('track_no'):
        actual_label = group['track_class'].iloc[0] if 'track_class' in group.columns else "UNKNOWN"
        pred_res = predict_single_track(group, model, le, scaler)

        if actual_label != "UNKNOWN":
            result_marker = '✅ Correct' if actual_label == pred_res['prediction'] else '❌ Wrong'
        else:
            result_marker = '🔍 Unverified'

        final_results.append({
            'Track ID': track_id,
            'Actual Class': actual_label,
            'Predicted Class': pred_res['prediction'],
            'Confidence': f"{pred_res['confidence']:.4f}",
            'Result': result_marker
        })
    return final_results

def visualize_results_table(results):
    df_res = pd.DataFrame(results)

    # Only calculate accuracy if we actually had ground truth labels
    if "UNKNOWN" not in df_res['Actual Class'].values:
        correct_count = len(df_res[df_res['Result'] == '✅ Correct'])
        total = len(df_res)
        accuracy = (correct_count / total) * 100 if total > 0 else 0
        acc_string = f" | Correct: {correct_count} | Accuracy: {accuracy:.2f}%"
    else:
        total = len(df_res)
        acc_string = " | Accuracy: N/A (No Ground Truth Provided)"

    print("\n" + "="*60)
    print(f" FINAL CLASSIFICATION REPORT ")
    print("="*60)
    print(f"Total Tracks: {total}{acc_string}")
    print("-" * 60)
    print(df_res.to_string(index=False))
    print("="*60)

    out_file = "prediction_comparison_results.csv"
    df_res.to_csv(out_file, index=False)
    print(f"\nDetailed results saved to '{out_file}'")

if __name__ == "__main__":
    # =====================================================================
    # CHANGE YOUR TEST FILE PATH HERE
    # =====================================================================
    TEST_FILE_PATH = "/content/radar_data.csv"

    if os.path.exists(TEST_FILE_PATH):
        results_list = process_test_file(TEST_FILE_PATH)
        if results_list:
            visualize_results_table(results_list)
    else:
        print(f"ERROR: Could not find the file '{TEST_FILE_PATH}'. Please check the path and try again.")

Loading trained model and testing against: /content/radar_data.csv
✓ Model and encoders loaded successfully
Extracting and bridging features...

 REAL-TIME STREAMING SIMULATION (Packet-by-Packet)

📡 Tracking ID 1 (Actual Target: 5)
  Packets received:  1 -> Predicted: CIVILIAN     | Confidence: 55.46% ⏳
  Packets received:  3 -> Predicted: FIGHTER_JET  | Confidence: 88.92% ⏳
  Packets received: 10 -> Predicted: FIGHTER_JET  | Confidence: 99.66% ⏳
  Packets received: 25 -> Predicted: FIGHTER_JET  | Confidence: 99.46% ⏳
  Packets received: 82 -> Predicted: CIVILIAN     | Confidence: 99.78% ⏳

Processing final full sequences for report...

 FINAL CLASSIFICATION REPORT 
Total Tracks: 1 | Correct: 0 | Accuracy: 0.00%
------------------------------------------------------------
 Track ID  Actual Class Predicted Class Confidence  Result
        1             5        CIVILIAN     0.9978 ❌ Wrong

Detailed results saved to 'prediction_comparison_results.csv'


**Improved version with transformers**

In [2]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 786.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 85.3 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


In [3]:
import numpy as np
import pandas as pd
import random
import joblib
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Bidirectional, GlobalAveragePooling1D, BatchNormalization, Conv1D, MaxPooling1D, Masking, concatenate, MultiHeadAttention, LayerNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# --- UNIFIED FEATURE EXTRACTOR ---
def build_features(df):
    df = df.copy()
    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)

    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)
    df['accel_x'] = df.groupby('track_no')['vel_x'].diff() / df['dt']
    df['accel_y'] = df.groupby('track_no')['vel_y'].diff() / df['dt']
    df['accel_z'] = df.groupby('track_no')['vel_z'].diff() / df['dt']
    df[['accel_x', 'accel_y', 'accel_z']] = df[['accel_x', 'accel_y', 'accel_z']].fillna(0)
    df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / df['dt']
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / df['dt']
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / df['dt']
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)

    df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['curvature'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2) / (df['range_mag'] + 1e-8)
    df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360

    def calc_turn(group):
        v = group[['vel_x', 'vel_y', 'vel_z']].values
        dt = group['dt'].values
        res = [0.0]
        for i in range(1, len(v)):
            n1, n2 = np.linalg.norm(v[i-1]), np.linalg.norm(v[i])
            if n1>0 and n2>0 and dt[i]>0:
                cos_ang = np.clip(np.dot(v[i-1], v[i])/(n1*n2), -1, 1)
                res.append(np.degrees(np.arccos(cos_ang))/dt[i])
            else: res.append(0.0)
        return np.array(res, dtype=np.float64)

    df['velocity_turn_rate'] = df.groupby('track_no').apply(calc_turn).reset_index(level=0, drop=True)

    cols_to_convert = ['accel_mag', 'velocity_turn_rate', 'z_ratio', 'dt']
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_turn'] = df.groupby('track_no')['velocity_turn_rate'].cummax()
    df['cum_max_z_ratio'] = df.groupby('track_no')['z_ratio'].cummax()
    df['cum_min_z_ratio'] = df.groupby('track_no')['z_ratio'].cummin()

    df['accel_mag_rolling_mean'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
    df['accel_mag_rolling_std'] = df.groupby('track_no')['accel_mag'].rolling(5, min_periods=1).std().reset_index(level=0, drop=True)

    def calc_dir_change(group):
        h = group['heading'].values
        res = [0.0]
        for i in range(1, len(h)):
            delta = abs((h[i] - h[i-1]) % 360)
            res.append(1.0 if min(delta, 360-delta) > 45 else 0.0)
        return np.array(res, dtype=np.float64)

    df['direction_change_rate'] = df.groupby('track_no').apply(calc_dir_change).reset_index(level=0, drop=True)
    df['direction_change_rate'] = pd.to_numeric(df['direction_change_rate'], errors='coerce').fillna(0.0)

    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0.0)

    return df.fillna(0.0)

# --- COMPLEX TRAINING GENERATOR ---
def generate_complex_trajectory(track_class, duration=80, dt=10.0):
    pos_x_list, pos_y_list, pos_z_list, vel_x_list, vel_y_list, vel_z_list = [], [], [], [], [], []
    track_no_list, track_class_list, ntp_sec_list = [], [], []

    track_id = random.randint(100000, 999999)
    spawn_radius = random.uniform(10000, 80000)
    spawn_angle = random.uniform(0, 2 * np.pi)
    pos_x, pos_y = spawn_radius * np.cos(spawn_angle), spawn_radius * np.sin(spawn_angle)

    if track_class == "CIVILIAN":
        pos_z, v_mag = np.random.uniform(5000, 12000), np.random.uniform(220, 310)
    elif track_class == "MISSILE":
        pos_z = np.random.uniform(50, 500) if random.random() < 0.3 else np.random.uniform(8000, 22000)
        v_mag = np.random.uniform(400, 1100)
    elif track_class == "HELICOPTER":
        pos_z, v_mag = np.random.uniform(20, 1500), np.random.uniform(0, 85)
    elif track_class == "FIGHTER_JET":
        pos_z, v_mag = np.random.uniform(500, 19000), np.random.uniform(250, 750)
    elif track_class == "SKIFF":
        pos_z, v_mag = 0.0, np.random.uniform(2, 28)

    angle = random.uniform(0, 2 * np.pi)
    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])

    start_time = 3991182426
    actual_elapsed_time, maneuver_timer, target_turn_rate = 0.0, 0, 0.0
    maneuver_type = "NONE"
    in_burst_loss = False

    for t in range(duration):
        if maneuver_timer <= 0:
            if track_class == "FIGHTER_JET" and random.random() < 0.15:
                maneuver_timer = random.randint(4, 10)
                maneuver_type = random.choice(["EVASIVE_TURN", "SPLIT_S_DIVE"])
                target_turn_rate = np.radians(random.uniform(-15, 15))
            elif track_class == "CIVILIAN" and random.random() < 0.10:
                maneuver_timer = random.randint(15, 30)
                maneuver_type = "HOLDING_PATTERN"
                target_turn_rate = np.radians(random.choice([-3, 3]))
            elif track_class == "HELICOPTER" and random.random() < 0.20:
                maneuver_timer = random.randint(3, 8)
                maneuver_type = random.choice(["POP_UP", "SUDDEN_HOVER"])
            elif track_class == "SKIFF" and random.random() < 0.25:
                maneuver_timer = random.randint(2, 6)
                maneuver_type = "ZIG_ZAG"
                target_turn_rate = np.radians(random.uniform(-25, 25))
            elif track_class == "MISSILE" and random.random() < 0.10:
                maneuver_timer = random.randint(3, 5)
                maneuver_type = "TERMINAL_POP_UP"
            else:
                maneuver_type = "CRUISE"

        if maneuver_timer > 0:
            maneuver_timer -= 1
            if maneuver_type in ["EVASIVE_TURN", "HOLDING_PATTERN", "ZIG_ZAG"]:
                c, s = np.cos(target_turn_rate * dt), np.sin(target_turn_rate * dt)
                vel[0], vel[1] = vel[0]*c - vel[1]*s, vel[0]*s + vel[1]*c
            elif maneuver_type == "SPLIT_S_DIVE":
                vel[2] -= 40 * dt; vel[0] *= 1.05; vel[1] *= 1.05
            elif maneuver_type == "POP_UP":
                vel[2] = 25; vel[0] *= 0.5; vel[1] *= 0.5
            elif maneuver_type == "SUDDEN_HOVER":
                vel *= 0.5
            elif maneuver_type == "TERMINAL_POP_UP":
                vel[2] = 50
        else:
            if track_class != "CIVILIAN": vel += np.random.normal(0, 1.5, 3)
            else: vel += np.random.normal(0, 0.2, 3); vel[2] *= 0.95

        if track_class == "SKIFF": vel[2] = 0.0

        pos_x += vel[0] * dt; pos_y += vel[1] * dt; pos_z += vel[2] * dt
        if pos_z < 0 and track_class != "SKIFF": pos_z, vel[2] = 0.0, 0.0
        if track_class == "SKIFF": pos_z = 0.0

        actual_elapsed_time += dt

        if in_burst_loss:
            if random.random() < 0.60: in_burst_loss = False
            else: continue
        else:
            if random.random() < 0.08:
                in_burst_loss = True
                continue

        noise_mult = max(1.0, (np.sqrt(pos_x**2 + pos_y**2 + pos_z**2) / 25000) ** 1.5)
        pos_x_list.append(pos_x + np.random.normal(0, 15 * noise_mult))
        pos_y_list.append(pos_y + np.random.normal(0, 15 * noise_mult))
        pos_z_list.append(abs(np.random.normal(0, 8 * noise_mult)) if track_class == "SKIFF" else pos_z + np.random.normal(0, 20 * noise_mult))
        vel_x_list.append(vel[0] + np.random.normal(0, 1.5 * noise_mult))
        vel_y_list.append(vel[1] + np.random.normal(0, 1.5 * noise_mult))
        vel_z_list.append(vel[2] + np.random.normal(0, 1.5 * noise_mult))

        track_no_list.append(track_id); track_class_list.append(track_class); ntp_sec_list.append(start_time + int(actual_elapsed_time))

    return pd.DataFrame({
        'pos_x': pos_x_list, 'pos_y': pos_y_list, 'pos_z': pos_z_list,
        'vel_x': vel_x_list, 'vel_y': vel_y_list, 'vel_z': vel_z_list,
        'track_no': track_no_list, 'track_class': track_class_list, 'ntp_sec': ntp_sec_list
    })

def generate_dataset(num_samples=250, duration=80):
    dfs = []
    for class_choice in ["FIGHTER_JET", "SKIFF", "MISSILE", "HELICOPTER", "CIVILIAN"]:
        for _ in range(num_samples):
            df = generate_complex_trajectory(class_choice, duration=duration)
            if df is not None and len(df) > 5: dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

# --- HYBRID CNN-BILSTM-TRANSFORMER ARCHITECTURE ---
def build_transformer_hybrid(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    masked = Masking(mask_value=0.0)(inputs)

    # 1. Local Pattern Extraction (Inception CNN)
    c1 = Conv1D(32, kernel_size=3, padding='same', activation='relu')(masked)
    c2 = Conv1D(32, kernel_size=5, padding='same', activation='relu')(masked)
    c3 = Conv1D(32, kernel_size=7, padding='same', activation='relu')(masked)
    x = concatenate([c1, c2, c3])
    x = BatchNormalization()(x)

    # 2. Sequential & Positional Context (Bi-LSTM)
    lstm_out = Bidirectional(LSTM(64, return_sequences=True, dropout=0.1))(x)

    # 3. Global Context (Transformer Multi-Head Attention)
    attention_out = MultiHeadAttention(num_heads=4, key_dim=32)(lstm_out, lstm_out)
    attention_out = LayerNormalization()(lstm_out + attention_out) # Residual connection

    # 4. Aggregation
    pooled = GlobalAveragePooling1D()(attention_out)

    # 5. Classification Head
    dense = Dense(64, activation='relu')(pooled)
    dense = BatchNormalization()(dense)
    dense = Dropout(0.4)(dense)
    outputs = Dense(num_classes, activation='softmax')(dense)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

if __name__ == "__main__":
    if not os.path.exists("trained_model"): os.makedirs("trained_model")
    print("Generating complex real-world training data...")
    df = generate_dataset(num_samples=300)
    print("Building memory features...")
    df_enhanced = build_features(df)

    feature_cols = [
        'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'vel_mag', 'range_mag', 'curvature',
        'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
        'cum_max_z_ratio', 'cum_min_z_ratio',
        'vertical_accel_ratio', 'speed_change_rate'
    ]

    scaler = StandardScaler()
    df_enhanced[feature_cols] = scaler.fit_transform(df_enhanced[feature_cols])
    joblib.dump(scaler, "trained_model/feature_scaler.pkl")

    seq_length = 50
    track_sequences, track_labels = [], []
    print("Building progressive sequences for early classification...")

    for track_id, group in df_enhanced.groupby('track_no'):
        data_matrix = group.sort_values('ntp_sec')[feature_cols].values

        for i in range(1, len(data_matrix) + 1):
            if i <= seq_length:
                window = data_matrix[:i]
                padded = np.pad(window, ((seq_length - len(window), 0), (0, 0)), 'constant')
                track_sequences.append(padded)
            else:
                window = data_matrix[i-seq_length:i]
                track_sequences.append(window)
            track_labels.append(group['track_class'].iloc[0])

    X, y = np.array(track_sequences), np.array(track_labels)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    joblib.dump(le, "trained_model/label_encoder.pkl")

    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
    model = build_transformer_hybrid((X.shape[1], X.shape[2]), len(le.classes_))

    callbacks = [EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
                 ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)]

    print("Training Transformer Hybrid Model...")
    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=64, callbacks=callbacks)
    model.save("trained_model/radar_target_classifier.keras")
    print("Training Complete!")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Generating complex real-world training data...
Building memory features...
Building progressive sequences for early classification...
Training Transformer Hybrid Model...
Epoch 1/25
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 102s 74ms/step - accuracy: 0.9489 - loss: 0.1275 - val_accuracy: 0.9749 - val_loss: 0.0574 - learning_rate: 0.0010
Epoch 2/25
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 97s 73ms/step - accuracy: 0.9742 - loss: 0.0643 - val_accuracy: 0.9822 - val_loss: 0.0401 - learning_rate: 0.0010
Epoch 3/25
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 97s 73ms/step - accuracy: 0.9787 - loss: 0.0537 - val_accuracy: 0.9821 - val_loss: 0.0409 - learning_rate: 0.0010
Epoch 4/25
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 97s 73ms/step - accuracy: 0.9824 - loss: 0.0438 - val_accuracy: 0.9846 - val_loss: 0.0333 - learning_rate: 0.0010
Epoch 5/25
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 97s 73ms/step - accuracy: 0.9836 - loss: 0.0398 - val_accuracy: 0.9889 - val_loss: 0.0289 - learning_rate: 0.0010
Epoch 6/25
1326/1326 ━━━━━━━━━━━━━━━━━━━━ 96s 73

In [5]:
import numpy as np
import pandas as pd
import random

def generate_complex_trajectory(track_class, duration=60, dt=10.0):
    pos_x_list, pos_y_list, pos_z_list = [], [], []
    vel_x_list, vel_y_list, vel_z_list = [], [], []
    track_no_list, track_class_list, ntp_sec_list = [], [], []

    track_id = random.randint(100000, 999999)

    # Spawn targets at varying ranges to test range-dependent noise
    spawn_radius = random.uniform(10000, 80000)
    spawn_angle = random.uniform(0, 2 * np.pi)
    pos_x = spawn_radius * np.cos(spawn_angle)
    pos_y = spawn_radius * np.sin(spawn_angle)

    # Overlapping boundary initializations
    if track_class == "CIVILIAN":
        pos_z = np.random.uniform(5000, 12000)
        v_mag = np.random.uniform(220, 310)
    elif track_class == "MISSILE":
        # 30% chance to be a "Sea Skimmer" missile (very low altitude)
        pos_z = np.random.uniform(50, 500) if random.random() < 0.3 else np.random.uniform(8000, 22000)
        v_mag = np.random.uniform(400, 1100)
    elif track_class == "HELICOPTER":
        pos_z = np.random.uniform(20, 1500)
        v_mag = np.random.uniform(0, 85) # Can start in a hover (0 m/s)
    elif track_class == "FIGHTER_JET":
        pos_z = np.random.uniform(500, 19000)
        v_mag = np.random.uniform(250, 750)
    elif track_class == "SKIFF":
        pos_z = 0.0
        v_mag = np.random.uniform(2, 28)

    angle = random.uniform(0, 2 * np.pi)
    vel = np.array([v_mag * np.cos(angle), v_mag * np.sin(angle), 0.0])

    start_time = 3991182426
    actual_elapsed_time = 0.0

    # State Machine Variables for Sustained Maneuvers
    maneuver_timer = 0
    maneuver_type = "NONE"
    target_turn_rate = 0.0

    # Sensor failure simulation variables
    in_burst_loss = False

    for t in range(duration):
        # ---------------------------------------------------
        # 1. SUSTAINED MANEUVER LOGIC (The "Real World" aspect)
        # ---------------------------------------------------
        if maneuver_timer <= 0:
            # Chance to start a new maneuver
            if track_class == "FIGHTER_JET" and random.random() < 0.15:
                maneuver_timer = random.randint(4, 10) # Lasts 40 to 100 seconds
                maneuver_type = random.choice(["EVASIVE_TURN", "SPLIT_S_DIVE"])
                target_turn_rate = np.radians(random.uniform(-15, 15)) # Deg per second

            elif track_class == "CIVILIAN" and random.random() < 0.10:
                maneuver_timer = random.randint(15, 30) # Long holding pattern
                maneuver_type = "HOLDING_PATTERN"
                target_turn_rate = np.radians(random.choice([-3, 3])) # Standard rate turn

            elif track_class == "HELICOPTER" and random.random() < 0.20:
                maneuver_timer = random.randint(3, 8)
                maneuver_type = random.choice(["POP_UP", "SUDDEN_HOVER"])

            elif track_class == "SKIFF" and random.random() < 0.25:
                maneuver_timer = random.randint(2, 6)
                maneuver_type = "ZIG_ZAG"
                target_turn_rate = np.radians(random.uniform(-25, 25))

            elif track_class == "MISSILE" and random.random() < 0.10:
                maneuver_timer = random.randint(3, 5)
                maneuver_type = "TERMINAL_POP_UP" # Pops up before diving
            else:
                maneuver_type = "CRUISE"

        # Execute active maneuver
        if maneuver_timer > 0:
            maneuver_timer -= 1

            if maneuver_type in ["EVASIVE_TURN", "HOLDING_PATTERN", "ZIG_ZAG"]:
                # Apply sustained turning over dt
                c, s = np.cos(target_turn_rate * dt), np.sin(target_turn_rate * dt)
                vel[0], vel[1] = vel[0]*c - vel[1]*s, vel[0]*s + vel[1]*c

            elif maneuver_type == "SPLIT_S_DIVE":
                vel[2] -= 40 * dt # Massive sustained dive
                vel[0] *= 1.05; vel[1] *= 1.05 # Accelerate in dive

            elif maneuver_type == "POP_UP":
                vel[2] = 25 # Rapid climb
                vel[0] *= 0.5; vel[1] *= 0.5 # Lose horizontal speed

            elif maneuver_type == "SUDDEN_HOVER":
                vel *= 0.5 # Bleed speed rapidly to 0

            elif maneuver_type == "TERMINAL_POP_UP":
                vel[2] = 50 # Sharp climb

        else:
            # Normal Cruising - Add slight natural drift
            if track_class != "CIVILIAN":
                vel += np.random.normal(0, 1.5, 3)
            else:
                vel += np.random.normal(0, 0.2, 3)
                vel[2] *= 0.95 # Civil flights tend to strictly self-level

        # Strict Boundaries
        if track_class == "SKIFF":
            vel[2] = 0.0

        # ---------------------------------------------------
        # 2. PHYSICS UPDATE
        # ---------------------------------------------------
        pos_x += vel[0] * dt
        pos_y += vel[1] * dt
        pos_z += vel[2] * dt

        # Hard ground floor
        if pos_z < 0 and track_class != "SKIFF":
            pos_z = 0.0
            vel[2] = 0.0
        if track_class == "SKIFF": pos_z = 0.0

        actual_elapsed_time += dt

        # ---------------------------------------------------
        # 3. BURST PACKET LOSS SIMULATION
        # ---------------------------------------------------
        if in_burst_loss:
            if random.random() < 0.60: # 60% chance to recover radar lock
                in_burst_loss = False
            else:
                continue # Skip appending! Model receives a massive time gap next packet.
        else:
            if random.random() < 0.08: # 8% chance to lose radar lock
                in_burst_loss = True
                continue

        # ---------------------------------------------------
        # 4. RANGE-DEPENDENT SENSOR NOISE
        # ---------------------------------------------------
        r_mag = np.sqrt(pos_x**2 + pos_y**2 + pos_z**2)
        noise_multiplier = max(1.0, (r_mag / 25000) ** 1.5) # Noise scales exponentially past 25km

        pos_x_noisy = pos_x + np.random.normal(0, 15 * noise_multiplier)
        pos_y_noisy = pos_y + np.random.normal(0, 15 * noise_multiplier)

        if track_class == "SKIFF":
            # Sea clutter / wave bounce
            pos_z_noisy = abs(np.random.normal(0, 8 * noise_multiplier))
        else:
            pos_z_noisy = pos_z + np.random.normal(0, 20 * noise_multiplier)

        vel_x_noisy = vel[0] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_y_noisy = vel[1] + np.random.normal(0, 1.5 * noise_multiplier)
        vel_z_noisy = vel[2] + np.random.normal(0, 1.5 * noise_multiplier)

        # ---------------------------------------------------
        # 5. RECORD DATA
        # ---------------------------------------------------
        timestamp = start_time + int(actual_elapsed_time)

        pos_x_list.append(pos_x_noisy)
        pos_y_list.append(pos_y_noisy)
        pos_z_list.append(pos_z_noisy)
        vel_x_list.append(vel_x_noisy)
        vel_y_list.append(vel_y_noisy)
        vel_z_list.append(vel_z_noisy)
        track_no_list.append(track_id)
        track_class_list.append(track_class)
        ntp_sec_list.append(timestamp)

    # Note: Because of burst drop, the lists might be shorter than `duration`. That's intended!
    df_data = {
        'pos_x': pos_x_list, 'pos_y': pos_y_list, 'pos_z': pos_z_list,
        'vel_x': vel_x_list, 'vel_y': vel_y_list, 'vel_z': vel_z_list,
        'track_no': track_no_list, 'track_class': track_class_list, 'ntp_sec': ntp_sec_list
    }
    return pd.DataFrame(df_data)

def generate_stress_test_dataset(num_samples=150, duration=80):
    dfs = []
    track_classes = ["FIGHTER_JET", "SKIFF", "MISSILE", "HELICOPTER", "CIVILIAN"]
    samples_per_class = num_samples // len(track_classes)

    for class_choice in track_classes:
        for _ in range(samples_per_class):
            df = generate_complex_trajectory(class_choice, duration=duration)
            if df is not None and len(df) > 5: # Only keep if at least 5 packets survived the burst loss
                dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

if __name__ == "__main__":
    print("Initializing Real-World Radar Stress Test Generator...")
    # Generating 150 tracks, tracked over 80 potential scans (800 seconds max)
    test_df = generate_stress_test_dataset(num_samples=150, duration=80)
    test_df.to_csv("test_radar_data.csv", index=False)
    print(f"Successfully generated advanced stress-test dataset with {len(test_df)} packets.")
    print("Features injected:")
    print(" - Burst packet loss (Time jumps > 10s)")
    # Print max dt to prove time jumps exist
    test_df['dt'] = test_df.groupby('track_no')['ntp_sec'].diff()
    max_jump = test_df['dt'].max()
    print(f" - Maximum sensor blackout period detected: {max_jump} seconds")
    print(" - Range-dependent exponentially scaling noise")
    print(" - Sustained Holding Patterns, Evasive Dives, and Hovering Maneuvers")

Initializing Real-World Radar Stress Test Generator...
Successfully generated advanced stress-test dataset with 10663 packets.
Features injected:
 - Burst packet loss (Time jumps > 10s)
 - Maximum sensor blackout period detected: 110.0 seconds
 - Range-dependent exponentially scaling noise
 - Sustained Holding Patterns, Evasive Dives, and Hovering Maneuvers


In [7]:
import pandas as pd
import numpy as np
import time
from tensorflow.keras.models import load_model
import joblib

# --- SMART PREPROCESSING BRIDGE ---
def robust_feature_extractor(df):
    """Calculates all advanced features, falling back gracefully on missing data."""
    df = df.copy()

    if 'track_no' not in df.columns: df['track_no'] = 1
    if 'ntp_sec' not in df.columns: df['ntp_sec'] = np.arange(len(df)) * 10.0

    df = df.sort_values(['track_no', 'ntp_sec']).reset_index(drop=True)
    df['dt'] = df.groupby('track_no')['ntp_sec'].diff().fillna(10.0).replace(0, 10.0)

    for axis in ['x', 'y', 'z']:
        if f'vel_{axis}' not in df.columns:
            df[f'vel_{axis}'] = df.groupby('track_no')[f'pos_{axis}'].diff() / df['dt']
            df[f'vel_{axis}'] = df[f'vel_{axis}'].fillna(0)

    for axis in ['x', 'y', 'z']:
        if f'accel_{axis}' not in df.columns:
            df[f'accel_{axis}'] = df.groupby('track_no')[f'vel_{axis}'].diff() / df['dt']
            df[f'accel_{axis}'] = df[f'accel_{axis}'].fillna(0)

    if 'accel_mag' not in df.columns:
        df['accel_mag'] = np.sqrt(df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)

    df['jerk_x'] = df.groupby('track_no')['accel_x'].diff() / df['dt']
    df['jerk_y'] = df.groupby('track_no')['accel_y'].diff() / df['dt']
    df['jerk_z'] = df.groupby('track_no')['accel_z'].diff() / df['dt']
    df[['jerk_x', 'jerk_y', 'jerk_z']] = df[['jerk_x', 'jerk_y', 'jerk_z']].fillna(0)
    df['jerk_mag'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)

    if 'vel_mag' not in df.columns: df['vel_mag'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2 + df['vel_z']**2)
    if 'range_mag' not in df.columns: df['range_mag'] = np.sqrt(df['pos_x']**2 + df['pos_y']**2 + df['pos_z']**2)
    if 'heading' not in df.columns: df['heading'] = np.degrees(np.arctan2(df['vel_y'], df['vel_x'])) % 360

    df['z_ratio'] = df['vel_z'] / (df['vel_mag'] + 1e-8)
    df['curvature'] = np.sqrt(df['vel_x']**2 + df['vel_y']**2) / (df['range_mag'] + 1e-8)
    df['vertical_accel_ratio'] = df['accel_z'] / (df['accel_mag'] + 1e-8)
    df['speed_change_rate'] = df.groupby('track_no')['vel_mag'].diff().fillna(0.0)

    def calc_turn(group):
        v = group[['vel_x', 'vel_y', 'vel_z']].values
        dt = group['dt'].values
        res = [0.0]
        for i in range(1, len(v)):
            n1, n2 = np.linalg.norm(v[i-1]), np.linalg.norm(v[i])
            if n1 > 0 and n2 > 0 and dt[i] > 0:
                cos_ang = np.clip(np.dot(v[i-1], v[i])/(n1*n2), -1, 1)
                res.append(np.degrees(np.arccos(cos_ang))/dt[i])
            else: res.append(0.0)
        return np.array(res, dtype=np.float64)

    df['velocity_turn_rate'] = df.groupby('track_no').apply(calc_turn).reset_index(level=0, drop=True)

    cols_to_convert = ['accel_mag', 'velocity_turn_rate', 'z_ratio', 'dt']
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    df['cum_max_accel'] = df.groupby('track_no')['accel_mag'].cummax()
    df['cum_max_turn'] = df.groupby('track_no')['velocity_turn_rate'].cummax()
    df['cum_max_z_ratio'] = df.groupby('track_no')['z_ratio'].cummax()
    df['cum_min_z_ratio'] = df.groupby('track_no')['z_ratio'].cummin()

    df = df.fillna(0.0)

    required_features = [
        'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
        'accel_mag', 'jerk_mag', 'z_ratio',
        'vel_mag', 'range_mag', 'curvature',
        'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
        'cum_max_z_ratio', 'cum_min_z_ratio',
        'vertical_accel_ratio', 'speed_change_rate'
    ]

    for col in required_features:
        if col not in df.columns:
            df[col] = 0.0

    return df

# --- STATEFUL TRACK MANAGER ---
class LiveTrackManager:
    def __init__(self, model_path="trained_model/radar_target_classifier.keras"):
        print("Loading Hybrid Transformer Model...")
        self.model = load_model(model_path, compile=False)
        self.le = joblib.load("trained_model/label_encoder.pkl")
        self.scaler = joblib.load("trained_model/feature_scaler.pkl")

        # This dictionary is the "Memory" of the live system
        self.active_tracks = {}

        self.feature_cols = [
            'dt', 'pos_z', 'vel_x', 'vel_y', 'vel_z',
            'accel_mag', 'jerk_mag', 'z_ratio',
            'vel_mag', 'range_mag', 'curvature',
            'velocity_turn_rate', 'cum_max_accel', 'cum_max_turn',
            'cum_max_z_ratio', 'cum_min_z_ratio',
            'vertical_accel_ratio', 'speed_change_rate'
        ]

    def process_packet(self, raw_packet):
        """
        Receives a single dictionary (representing a UDP radar packet).
        Returns the updated prediction for that track.
        """
        tid = raw_packet['track_no']

        # 1. Update Track Memory
        if tid not in self.active_tracks:
            self.active_tracks[tid] = []

        self.active_tracks[tid].append(raw_packet)

        # Keep history from growing forever to save memory,
        # but keep enough to ensure cumulative features stay accurate.
        if len(self.active_tracks[tid]) > 200:
            self.active_tracks[tid] = self.active_tracks[tid][-200:]

        # 2. Extract Features dynamically
        track_df = pd.DataFrame(self.active_tracks[tid])
        features_df = robust_feature_extractor(track_df)

        # 3. Predict on the newly generated sequence
        try:
            features = features_df[self.feature_cols].values
            features_scaled = self.scaler.transform(features)

            seq_length = 50
            if len(features_scaled) >= seq_length:
                input_data = features_scaled[-seq_length:].reshape(1, seq_length, -1)
            else:
                padded = np.pad(features_scaled, ((seq_length - len(features_scaled), 0), (0, 0)), 'constant')
                input_data = padded.reshape(1, seq_length, -1)

            pred_probs = self.model.predict(input_data, verbose=0)[0]
            predicted_class_idx = np.argmax(pred_probs)
            confidence = np.max(pred_probs)
            predicted_label = self.le.inverse_transform([predicted_class_idx])[0]

            packet_count = len(self.active_tracks[tid])
            return {'track_id': tid, 'prediction': predicted_label, 'confidence': float(confidence), 'packets_received': packet_count}

        except Exception as e:
            return {'track_id': tid, 'prediction': f'ERROR: {str(e)}', 'confidence': 0.0, 'packets_received': 0}

if __name__ == "__main__":
    TEST_FILE_PATH = "/content/radar_data.csv"

    manager = LiveTrackManager()

    # We load the CSV and convert it to a list of dicts to mimic a live network queue
    print(f"\nSimulating live packet stream from {TEST_FILE_PATH}...")
    raw_stream_df = pd.read_csv(TEST_FILE_PATH).dropna(subset=['track_no'])

    # Sort purely by time to completely scramble the track numbers, mimicking a real multiplexed feed
    raw_stream_df = raw_stream_df.sort_values('ntp_sec')
    live_queue = raw_stream_df.to_dict('records')

    print("\n" + "="*70)
    print(" LIVE MULTIPLEXED DATA FEED SIMULATION")
    print("="*70)

    # To avoid printing 10,000 lines, we will only track a few specific targets
    target_ids_to_watch = list(raw_stream_df['track_no'].unique()[:3])

    for packet in live_queue:
        result = manager.process_packet(packet)

        # Only print updates for our watched targets to keep the console clean
        if result['track_id'] in target_ids_to_watch:
            actual_class = packet.get('track_class', 'UNKNOWN')
            status = "✅" if result['prediction'] == actual_class else ("⏳" if actual_class != "UNKNOWN" else "🔍")

            print(f"📡 [Time: {packet['ntp_sec']}] Track {result['track_id']} (Actual: {actual_class})")
            print(f"   -> Packets buffered: {result['packets_received']:02d} | Output: {result['prediction']:12s} | Confidence: {result['confidence']:.2%} {status}\n")

            # Optional: Slow down the loop slightly to visualize the "live" aspect
            # time.sleep(0.05)

Loading Hybrid Transformer Model...

Simulating live packet stream from /content/radar_data.csv...

 LIVE MULTIPLEXED DATA FEED SIMULATION
📡 [Time: 3991182426] Track 1 (Actual: 5)
   -> Packets buffered: 01 | Output: FIGHTER_JET  | Confidence: 71.78% ⏳

📡 [Time: 3991182434] Track 1 (Actual: 5)
   -> Packets buffered: 02 | Output: FIGHTER_JET  | Confidence: 71.68% ⏳

📡 [Time: 3991182444] Track 1 (Actual: 5)
   -> Packets buffered: 03 | Output: FIGHTER_JET  | Confidence: 78.62% ⏳

📡 [Time: 3991182454] Track 1 (Actual: 5)
   -> Packets buffered: 04 | Output: FIGHTER_JET  | Confidence: 96.54% ⏳

📡 [Time: 3991182464] Track 1 (Actual: 5)
   -> Packets buffered: 05 | Output: FIGHTER_JET  | Confidence: 97.37% ⏳

📡 [Time: 3991182474] Track 1 (Actual: 5)
   -> Packets buffered: 06 | Output: FIGHTER_JET  | Confidence: 99.75% ⏳

📡 [Time: 3991182484] Track 1 (Actual: 5)
   -> Packets buffered: 07 | Output: FIGHTER_JET  | Confidence: 96.50% ⏳

📡 [Time: 3991182494] Track 1 (Actual: 5)
   -> Packets b

# ***Integration into C***

In [1]:
!pip install tf2onnx onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 100.1 MB/s eta 0:00:00


In [3]:
import tensorflow as tf
import tf2onnx
import os

def export_to_onnx():
    print("Loading Keras model...")
    model = tf.keras.models.load_model("trained_model/radar_target_classifier.keras", compile=False)

    # Define the exact input shape your model expects.
    # None = Batch size (dynamic), 50 = Sequence Length, 18 = Features
    input_signature = [tf.TensorSpec([None, 50, 18], tf.float32, name='input')]

    onnx_model_path = "trained_model/radar_classifier.onnx"

    print("Converting to ONNX format...")
    model_proto, _ = tf2onnx.convert.from_keras(
        model,
        input_signature=input_signature,
        opset=13, # Standard operations set
        output_path=onnx_model_path
    )

    print(f"Success! Model exported to: {onnx_model_path}")
    print("Integration file ready for C code!")

if __name__ == "__main__":
    export_to_onnx()

Loading Keras model...
Converting to ONNX format...


Success! Model exported to: trained_model/radar_classifier.onnx
Integration file ready for C code!


In [6]:
!pip install onnxruntime
import numpy as np
import tensorflow as tf
import onnxruntime as ort

# 1. Create a dummy radar input tensor (1 batch, 50 sequence steps, 18 features)
dummy_input = np.random.randn(1, 50, 18).astype(np.float32)

# 2. Get prediction from Keras model
keras_model = tf.keras.models.load_model("trained_model/radar_target_classifier.keras", compile=False)
keras_pred = keras_model.predict(dummy_input, verbose=0)

# 3. Get prediction from exported ONNX model using ONNX Runtime
onnx_session = ort.InferenceSession("trained_model/radar_classifier.onnx")
input_name = onnx_session.get_inputs()[0].name
onnx_pred = onnx_session.run(None, {input_name: dummy_input})[0]

# 4. Compare outputs
diff = np.max(np.abs(keras_pred - onnx_pred))
print(f"Max Absolute Difference between Keras and ONNX: {diff:.8f}")

if diff < 1e-4:
    print("✅ VERIFIED: The ONNX model produces mathematically identical predictions!")
else:
    print("❌ Discrepancy detected.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 122.1 MB/s eta 0:00:00
Max Absolute Difference between Keras and ONNX: 0.00000000
✅ VERIFIED: The ONNX model produces mathematically identical predictions!


***Code to deliver to tracker***

This is the exact deliverable you need to hand off to the C/C++ integration team.

To bridge the gap between Python and C, the C team needs two things:

The Scaler Values: The hardcoded mean and variance numbers from your Python StandardScaler.

The Stateful Feature Extractor: A C struct that remembers previous packets and calculates the exact 18 features (including derivatives like jerk and cumulative maxes) dynamically.

Here is the complete blueprint you can share with them.

Part 1: Exporting the Python Scaler to C Arrays
Before you hand off the code, you need to extract the normalization values from your feature_scaler.pkl so the C code can use them. Run this quick Python snippet in your notebook:

In [7]:
import joblib
import numpy as np

# Load the trained scaler
scaler = joblib.load("trained_model/feature_scaler.pkl")

# Generate C-style array strings
print("Copy these arrays into the C code:\n")
print(f"const float SCALER_MEAN[18] = {{{', '.join(map(str, scaler.mean_))}}};")
print(f"const float SCALER_SCALE[18] = {{{', '.join(map(str, scaler.scale_))}}};")

Copy these arrays into the C code:

const float SCALER_MEAN[18] = {11.275016504762803, 8614.945337894602, 7.246215966614835, 2.4859739265454106, 10.042536143509343, 13.216322590864808, 1.8625562602113084, 0.11405606813637222, 413.95375879296046, 131602.1421680016, 0.0038966063555973436, 0.0, 29.834302672865107, 0.0, 0.3453915613031254, -0.34390011109961593, -0.0034169194991615346, 3.2872780370091164};
const float SCALER_SCALE[18] = {5.21781577113403, 11780.838735045161, 447.5957937539523, 466.7001761051196, 69.4763383667835, 32.91351716052653, 5.179034119906, 0.3643594354447169, 501.7766797901309, 164583.84756785867, 0.006366845993239587, 1.0, 54.31523357826823, 1.0, 0.3777993407959899, 0.3651378804521134, 0.48925845328793766, 86.0800242133189};


Part 2: The C/C++ Tracker Integration Code
This is the code your integration team will add to the tracker. It is written in clean, embedded-friendly C (fully compatible with C++) and requires zero dynamic memory allocation (malloc), making it blazingly fast.

It maintains a circular memory buffer for each target, calculates the complex physics, applies the normalization, and formats the final [50 * 18] array exactly how the ONNX model expects it.

In [ ]:
#include <stdio.h>
#include <math.h>
#include <string.h>

#define SEQ_LENGTH 50
#define NUM_FEATURES 18
#define PI 3.14159265358979323846

// =======================================================================
// PASTE THE PYTHON SCALER VALUES HERE
// =======================================================================
const float SCALER_MEAN[NUM_FEATURES] = {0.0, 0.0, 0.0};  // Paste from Python
const float SCALER_SCALE[NUM_FEATURES] = {1.0, 1.0, 1.0}; // Paste from Python

// Structure representing the raw data coming from the radar
typedef struct {
    double ntp_sec;
    float pos_x;
    float pos_y;
    float pos_z;
    // Note: If radar provides vel, set these. If not, pass 0 and the code calculates it.
    float vel_x;
    float vel_y;
    float vel_z;
} RawRadarPacket;

// The Stateful Memory Buffer for a single Track
typedef struct {
    // Sequence buffer for the ONNX model
    float feature_sequence[SEQ_LENGTH][NUM_FEATURES];
    int packet_count; // How many packets received so far (max 50)

    // Physics Memory
    double prev_ntp_sec;
    float prev_pos[3];
    float prev_vel[3];
    float prev_accel[3];
    float prev_vel_mag;

    // Cumulative Memory Features
    float cum_max_accel;
    float cum_max_turn;
    float cum_max_z_ratio;
    float cum_min_z_ratio;

    // Rolling memory for 5-step window
    float accel_history[5];
    int accel_hist_idx;
    int accel_hist_count;
} TrackState;

// Initialize a new track state (Call this when a new Track ID is detected)
void init_track_state(TrackState* state) {
    memset(state, 0, sizeof(TrackState));
    state->cum_min_z_ratio = 999999.0f; // Start high for cum_min
}

// Float clipping helper
float clip(float val, float min_val, float max_val) {
    if (val < min_val) return min_val;
    if (val > max_val) return max_val;
    return val;
}

// =======================================================================
// THE SMART PREPROCESSING BRIDGE (C VERSION)
// =======================================================================
void process_packet_and_get_onnx_input(TrackState* state, const RawRadarPacket* pkt, float* onnx_input_buffer) {

    float features[NUM_FEATURES] = {0};

    // 1. Calculate Time Delta (dt)
    float dt = 10.0f;
    if (state->packet_count > 0) {
        dt = (float)(pkt->ntp_sec - state->prev_ntp_sec);
        if (dt <= 0.0f) dt = 10.0f; // Fallback
    }

    // 2. Kinematics Derivation
    float vel[3] = {pkt->vel_x, pkt->vel_y, pkt->vel_z};

    // Fallback: If radar doesn't provide velocity, derive it from position
    if (vel[0] == 0 && vel[1] == 0 && vel[2] == 0 && state->packet_count > 0) {
        vel[0] = (pkt->pos_x - state->prev_pos[0]) / dt;
        vel[1] = (pkt->pos_y - state->prev_pos[1]) / dt;
        vel[2] = (pkt->pos_z - state->prev_pos[2]) / dt;
    }

    float accel[3] = {0};
    if (state->packet_count > 0) {
        accel[0] = (vel[0] - state->prev_vel[0]) / dt;
        accel[1] = (vel[1] - state->prev_vel[1]) / dt;
        accel[2] = (vel[2] - state->prev_vel[2]) / dt;
    }

    float accel_mag = sqrtf(accel[0]*accel[0] + accel[1]*accel[1] + accel[2]*accel[2]);

    float jerk[3] = {0};
    if (state->packet_count > 0) {
        jerk[0] = (accel[0] - state->prev_accel[0]) / dt;
        jerk[1] = (accel[1] - state->prev_accel[1]) / dt;
        jerk[2] = (accel[2] - state->prev_accel[2]) / dt;
    }
    float jerk_mag = sqrtf(jerk[0]*jerk[0] + jerk[1]*jerk[1] + jerk[2]*jerk[2]);

    // 3. Magnitudes & Ratios
    float vel_mag = sqrtf(vel[0]*vel[0] + vel[1]*vel[1] + vel[2]*vel[2]);
    float range_mag = sqrtf(pkt->pos_x*pkt->pos_x + pkt->pos_y*pkt->pos_y + pkt->pos_z*pkt->pos_z);

    float z_ratio = vel[2] / (vel_mag + 1e-8f);
    float curvature = sqrtf(vel[0]*vel[0] + vel[1]*vel[1]) / (range_mag + 1e-8f);
    float vertical_accel_ratio = accel[2] / (accel_mag + 1e-8f);
    float speed_change_rate = (state->packet_count > 0) ? (vel_mag - state->prev_vel_mag) : 0.0f;

    // 4. Velocity Turn Rate
    float turn_rate = 0.0f;
    if (state->packet_count > 0 && vel_mag > 0 && state->prev_vel_mag > 0) {
        float dot = state->prev_vel[0]*vel[0] + state->prev_vel[1]*vel[1] + state->prev_vel[2]*vel[2];
        float cos_ang = clip(dot / (state->prev_vel_mag * vel_mag), -1.0f, 1.0f);
        turn_rate = (acosf(cos_ang) * (180.0f / PI)) / dt;
    }

    // 5. Cumulative Memory Features
    if (accel_mag > state->cum_max_accel) state->cum_max_accel = accel_mag;
    if (turn_rate > state->cum_max_turn) state->cum_max_turn = turn_rate;
    if (z_ratio > state->cum_max_z_ratio) state->cum_max_z_ratio = z_ratio;
    if (z_ratio < state->cum_min_z_ratio) state->cum_min_z_ratio = z_ratio;

    // 6. Rolling Window (Accel Mag)
    state->accel_history[state->accel_hist_idx] = accel_mag;
    state->accel_hist_idx = (state->accel_hist_idx + 1) % 5;
    if (state->accel_hist_count < 5) state->accel_hist_count++;

    float roll_mean = 0.0f;
    for(int i=0; i < state->accel_hist_count; i++) roll_mean += state->accel_history[i];
    roll_mean /= state->accel_hist_count;

    float roll_std = 0.0f;
    if (state->accel_hist_count > 1) {
        float variance = 0.0f;
        for(int i=0; i < state->accel_hist_count; i++) {
            variance += powf(state->accel_history[i] - roll_mean, 2);
        }
        roll_std = sqrtf(variance / (state->accel_hist_count - 1)); // Pandas uses N-1 for std
    }

    // 7. Assign to Feature Array (MUST EXACTLY MATCH PYTHON COLUMN ORDER)
    features[0] = dt;
    features[1] = pkt->pos_z;
    features[2] = vel[0];
    features[3] = vel[1];
    features[4] = vel[2];
    features[5] = accel_mag;
    features[6] = jerk_mag;
    features[7] = z_ratio;
    features[8] = vel_mag;
    features[9] = range_mag;
    features[10] = curvature;
    features[11] = turn_rate;
    features[12] = state->cum_max_accel;
    features[13] = state->cum_max_turn;
    features[14] = state->cum_max_z_ratio;
    features[15] = state->cum_min_z_ratio;
    features[16] = vertical_accel_ratio;
    features[17] = speed_change_rate;

    // 8. Scale Features using Python StandardScaler values
    for (int i = 0; i < NUM_FEATURES; i++) {
        features[i] = (features[i] - SCALER_MEAN[i]) / SCALER_SCALE[i];
    }

    // 9. Update Track Sequence Buffer
    // Shift elements left if we are at max capacity
    if (state->packet_count >= SEQ_LENGTH) {
        memmove(state->feature_sequence[0], state->feature_sequence[1], (SEQ_LENGTH - 1) * NUM_FEATURES * sizeof(float));
        memcpy(state->feature_sequence[SEQ_LENGTH - 1], features, NUM_FEATURES * sizeof(float));
    } else {
        memcpy(state->feature_sequence[state->packet_count], features, NUM_FEATURES * sizeof(float));
        state->packet_count++;
    }

    // 10. Save Current State for Next Packet
    state->prev_ntp_sec = pkt->ntp_sec;
    state->prev_pos[0] = pkt->pos_x; state->prev_pos[1] = pkt->pos_y; state->prev_pos[2] = pkt->pos_z;
    state->prev_vel[0] = vel[0];     state->prev_vel[1] = vel[1];     state->prev_vel[2] = vel[2];
    state->prev_accel[0] = accel[0]; state->prev_accel[1] = accel[1]; state->prev_accel[2] = accel[2];
    state->prev_vel_mag = vel_mag;

    // 11. Format the Final ONNX Input Array (Pad with 0s at the front for early packets)
    memset(onnx_input_buffer, 0, SEQ_LENGTH * NUM_FEATURES * sizeof(float)); // Zero padding
    int padding_offset = (SEQ_LENGTH - state->packet_count) * NUM_FEATURES;
    memcpy(onnx_input_buffer + padding_offset, state->feature_sequence, state->packet_count * NUM_FEATURES * sizeof(float));
}